In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import mpl_axes_aligner

In [ ]:
#Functions to make life simpler
def pca_func(data: pd.DataFrame, title_suffix: str = "") -> PCA:
    """
    Perform PCA on standardized data and plot explained variance.
    Returns the fitted PCA object and the PCA scores.
    """
    pca = PCA()
    pca_out = pca.fit_transform(data)

    # Create new figure for explained variance
    plt.figure(figsize=(6, 4))
    plt.plot(np.arange(1, min(10, data.shape[1])+1),
             pca.explained_variance_ratio_[:min(10, data.shape[1])], marker='o')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance')
    plt.title(f'PCA Explained Variance{title_suffix}')
    plt.tight_layout()

    # Print explained variance table
    summary = pd.DataFrame({
        'Explained Variance Ratio': pca.explained_variance_ratio_,
        'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
    })
    print(f"\nExplained Variance Summary{title_suffix}")
    print(summary)

    return pca, pd.DataFrame(
        pca_out,
        columns=[f'PC{i}' for i in range(1, pca_out.shape[1] + 1)],
        index=data.index
    )


In [ ]:
# There is no biplot function in sklearn, so we create a simple one ourselves
#function to produce biplot
def biplot(df_scores: pd.DataFrame, df_loadings: pd.DataFrame, pca: PCA,
           title: str = "Biplot") -> plt.Axes:
    """
    Create a PCA biplot showing both score points and variable loadings.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(df_scores.PC1, df_scores.PC2, color='b', alpha=0.7)

    expl_var = 100 * pca.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({expl_var[0]:.1f}% explained var.)", fontsize=10)
    ax.set_ylabel(f"PC2 ({expl_var[1]:.1f}% explained var.)", fontsize=10)
   
    # Label sample points (parties)
    for i, name in enumerate(df_scores.index):
        x = df_scores.iloc[i, 0]
        y = df_scores.iloc[i, 1]
        ax.text(x, y, name, fontsize=9, color='blue', alpha=0.8)

    # Second set of axes for loadings
    ax2 = ax.twinx().twiny()
    font = {'color': 'g', 'weight': 'bold', 'size': 10}

    # Plot loading vectors
    for col in df_loadings.columns:
        tipx = df_loadings.loc['PC1', col]
        tipy = df_loadings.loc['PC2', col]
        ax2.arrow(0, 0, tipx, tipy, color='r', alpha=0.5, length_includes_head=True)
        ax2.text(tipx * 1.07, tipy * 1.07, col, fontdict=font, ha='center', va='center')


    # Align axes and keep aspect square
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)
    ax.set_aspect('equal', adjustable='datalim')
    ax2.set_aspect('equal', adjustable='datalim')

    plt.title(title)
    plt.tight_layout()
    return ax


In [ ]:
# Load and prepare data
dfr = pd.read_csv("data/Stemwijzer2023.csv", sep=',')
parties = dfr.iloc[:, 0]
df = dfr.iloc[:, 1:31]  # select only statement columns

scaler = StandardScaler()
dfs = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns,
    index=parties
)

In [ ]:
# Perform PCA and biplot
pca, df_scores = pca_func(dfs, " Stemwijzer")
df_loadings = pd.DataFrame(
    pca.components_,
    columns=dfs.columns,
    index=df_scores.columns
)

In [ ]:
# Draw main biplot
ax = biplot(df_scores, df_loadings, pca, title="Stemwijzer - PCA Biplot")
plt.show()

In [ ]:

# -----------------------------
# Simulate "agree with all statements" respondent

# Create a vector of agreement (value = 2 for all statements)
agree_vector = np.full(dfs.shape[1], 2.0)

# Center it by subtracting column means of unscaled data
gem = df.mean().values
agree_centered = agree_vector - gem

# Project into PCA space (first 2 components)
agree_pca = agree_centered @ pca.components_[:2, :].T

# Plot again with this respondent added
ax = biplot(df_scores, df_loadings, pca,
            title="Stemwijzer - PCA Biplot with 'Agree All'")
ax.scatter(agree_pca[0], agree_pca[1], color='red', s=150, label='Agree (all 2)')
ax.legend()
plt.show()